In [121]:
from scipy.io import loadmat
import numpy as np
data_train=loadmat('Data/data_train.mat')
data_test=loadmat('Data/data_test.mat')
label_train=loadmat('Data/label_train.mat')

print("data:",data_train.keys())
print("label:",label_train.keys())

X_train=np.array(data_train['data_train'])
y=np.array(label_train['label_train'])
X_test=np.array(data_test['data_test'])

print("X shape:",X_train.shape)
print("y shape:", y.shape)
print(X_test.shape)
print(np.unique(y))

data: dict_keys(['__header__', '__version__', '__globals__', 'data_train'])
label: dict_keys(['__header__', '__version__', '__globals__', 'label_train'])
X shape: (577, 5)
y shape: (577, 1)
(23, 5)
[-1  1]


In [122]:
class FisherLD_Classifier:
    def __init__(self):
        pass

    def fit(self,X,y):
        self.classes=np.unique(y)
        self.datas=[]
        self.mus=[]
        self.covs=[]
        n_classes=len(self.classes)
        if n_classes>2: 
            print('Please input training data with 2 classes\n')
            return None
        for c in self.classes:
            y=y.ravel()
            X_c=X[y==c]
            mu_c=np.mean(X_c,axis=0)
            cov_c=np.cov(X_c, rowvar=False) * (len(X_c) - 1)
            self.datas.append(X_c)
            self.mus.append(mu_c)
            self.covs.append(cov_c)
        S_w=self.covs[0]+self.covs[1]
        self.w=np.linalg.inv(S_w).dot(self.mus[0]-self.mus[1])
        self.b=-0.5*np.dot(self.w.T,self.mus[0]+self.mus[1])

    def predict(self,X):
        g = np.dot(X, self.w) + self.b
        y_pred = np.where(g > 0, self.classes[0], self.classes[1])
        return y_pred
    
    def evaluate(self,X_test,y_test):
        y_pred=self.predict(X_test)
        accuracy = np.mean(y_pred == y_test.ravel())
        return accuracy
    
    def get_w(self): return self.w
    def get_b(self): return self.b
    def get_classes(self): return self.classes

        



In [123]:
classifier=FisherLD_Classifier()
classifier.fit(X_train,y)
y_pred=classifier.predict(X_test)
w=classifier.get_w()
b=classifier.get_b()
accuracy=classifier.evaluate(X_train,y)
print(y_pred)
print("Prediction labels:")
print(y_pred)
print("W:",w)
print("b",b)
print("Training accuracy:", accuracy)


[-1 -1 -1 -1 -1 -1 -1 -1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1]
Prediction labels:
[-1 -1 -1 -1 -1 -1 -1 -1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1]
W: [0.00178602 0.0034729  0.00194673 0.00168257 0.00265808]
b -0.008700458474367305
Training accuracy: 0.9324090121317158
